In [ ]:
import os
from pathlib import Path
from src.spark_session import create_spark_session

local_data = os.getenv("FASTFOOD_LOCAL_DATA")
orders_path = (Path(local_data).resolve() / "orders_history.jsonl").as_uri() if local_data else "s3a://lake/bronze/history/orders_history.jsonl"
restaurants_path = (Path(local_data).resolve() / "restaurants.json").as_uri() if local_data else "s3a://lake/bronze/reference/restaurants.json"

spark = create_spark_session("FastFood-RDD", enable_s3=not local_data)
sc = spark.sparkContext

In [ ]:
import json

path = orders_path

raw_rdd = sc.textFile(path)

orders_rdd = raw_rdd.map(json.loads).cache()
expected_events = raw_rdd.count()
assert expected_events > 0

print("Nombre d'événements :", orders_rdd.count())
print("Partitions :", orders_rdd.getNumPartitions())
print("Exemple :", orders_rdd.first())

In [ ]:
delivered_rdd = (
    orders_rdd
    .filter(lambda order: order["status"] == "DELIVERED")
    .map(lambda order: (
        order["order_id"],
        order
    ))
    .reduceByKey(lambda a, b: a)
    .values().cache()
)

revenue_by_city = (
    delivered_rdd
    .map(lambda order: (
        order["city"],
        float(order["amount_mad"])
    ))
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
)

for city, revenue in revenue_by_city.collect():
    print(f"{city}: {revenue:.2f} MAD")

In [ ]:
city_stats = (
    delivered_rdd
    .map(lambda order: (
        order["city"],
        (float(order["amount_mad"]), 1)
    ))
    .reduceByKey(
        lambda a, b: (
            a[0] + b[0],
            a[1] + b[1]
        )
    )
)

average_basket = city_stats.mapValues(
    lambda value: value[0] / value[1]
)

for city, average in average_basket.sortByKey().collect():
    print(f"{city}: {average:.2f} MAD")

In [ ]:
top_restaurants = (
    delivered_rdd.map(lambda order: (order["restaurant_id"], 1))
    .reduceByKey(lambda a, b: a + b)
    .takeOrdered(5, key=lambda item: (-item[1], item[0]))
)
print("Top 5 par commandes livrées")
for restaurant_id, count in top_restaurants:
    print(restaurant_id, count)

In [ ]:
total_events = orders_rdd.count()
delivered_events = orders_rdd.filter(
    lambda o: o["status"] == "DELIVERED"
).count()
unique_delivered_orders = delivered_rdd.count()

print("Événements totaux :", total_events)
print("Événements DELIVERED :", delivered_events)
print("Commandes livrées uniques :", unique_delivered_orders)
print("Doublons DELIVERED :", delivered_events - unique_delivered_orders)

assert total_events == expected_events
assert unique_delivered_orders <= delivered_events

In [ ]:
from operator import add

status_counts = (
    orders_rdd
    .map(lambda order: (order["status"], 1))
    .reduceByKey(add)
    .sortByKey()
)

for status, count in status_counts.collect():
    print(f"{status}: {count}")

assert sum(count for _, count in status_counts.collect()) == expected_events

print("STATUS ANALYSIS PASSED")

In [ ]:
# Identité approximative d'un événement :
# order_id + status + event_time

event_keys = orders_rdd.map(
    lambda o: (
        o["order_id"],
        o["status"],
        o["event_time"]
    )
)

total_events = orders_rdd.count()
unique_events = event_keys.distinct().count()

duplicate_events = total_events - unique_events

print("Total events:", total_events)
print("Unique events:", unique_events)
print("Duplicate events:", duplicate_events)
print(
    "Duplicate rate:",
    round(duplicate_events / total_events * 100, 2),
    "%"
)

assert duplicate_events >= 0

In [ ]:
import time

city_amounts = delivered_rdd.map(
    lambda o: (
        o["city"],
        float(o["amount_mad"])
    )
).cache()

city_amounts.count()

# Méthode 1 : reduceByKey
start = time.perf_counter()

result_reduce = dict(
    city_amounts
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

time_reduce = time.perf_counter() - start

# Méthode 2 : groupByKey
start = time.perf_counter()

result_group = dict(
    city_amounts
    .groupByKey()
    .mapValues(lambda values: sum(values))
    .collect()
)

time_group = time.perf_counter() - start

print("reduceByKey:", round(time_reduce, 4), "seconds")
print("groupByKey:", round(time_group, 4), "seconds")

for city in result_reduce:
    assert abs(
        result_reduce[city] - result_group[city]
    ) < 0.01

print("RESULTS MATCH")

city_amounts.unpersist()

In [ ]:
print("Original partitions:", orders_rdd.getNumPartitions())

def inspect_partition(index, iterator):
    import socket

    count = sum(1 for _ in iterator)

    yield (
        index,
        socket.gethostname(),
        count
    )

partition_info = (
    orders_rdd
    .mapPartitionsWithIndex(inspect_partition)
    .collect()
)

for partition, worker, count in partition_info:
    print(
        f"Partition {partition} | "
        f"Worker: {worker} | "
        f"Records: {count}"
    )

assert sum(row[2] for row in partition_info) == expected_events

In [ ]:
import time

start = time.perf_counter()

orders_4p = orders_rdd.repartition(4).cache()

count = orders_4p.count()

elapsed = time.perf_counter() - start

print("Before:", orders_rdd.getNumPartitions())
print("After:", orders_4p.getNumPartitions())
print("Records:", count)
print("Elapsed:", round(elapsed, 4), "seconds")

assert orders_4p.getNumPartitions() == 4
assert count == expected_events

In [ ]:
# Réduire de 4 à 2 partitions sans shuffle
coalesced_rdd = orders_4p.coalesce(2)

# Réduire de 4 à 2 partitions avec shuffle
repartitioned_rdd = orders_4p.repartition(2)

print("Original:", orders_4p.getNumPartitions())
print("Coalesce:", coalesced_rdd.getNumPartitions())
print("Repartition:", repartitioned_rdd.getNumPartitions())

assert coalesced_rdd.count() == expected_events
assert repartitioned_rdd.count() == expected_events

print("PARTITION TEST PASSED")

In [ ]:
print("=== Original RDD ===")
print(orders_rdd.toDebugString().decode("utf-8"))

print("\n=== Revenue RDD ===")
print(revenue_by_city.toDebugString().decode("utf-8"))

print("\n=== Repartitioned RDD ===")
print(orders_4p.toDebugString().decode("utf-8"))

In [ ]:
import time

def benchmark(rdd, label, runs=3):
    durations = []

    for i in range(runs):
        start = time.perf_counter()

        count = rdd.count()

        duration = time.perf_counter() - start
        durations.append(duration)

        print(
            f"{label} | Run {i+1} | "
            f"Count: {count} | "
            f"Time: {duration:.4f}s"
        )

    return sum(durations) / len(durations)


avg_original = benchmark(
    orders_rdd,
    "Original"
)

avg_cached = benchmark(
    orders_4p,
    "Cached 4 partitions"
)

print("\nAverage original:", round(avg_original, 4))
print("Average cached:", round(avg_cached, 4))

In [ ]:
assert orders_rdd.count() == expected_events
assert orders_4p.count() == expected_events

assert duplicate_events >= 0

assert sum(
    count for _, count in status_counts.collect()
) == expected_events

assert len(result_reduce) == len(result_group)

orders_4p.unpersist()

print("=" * 45)
print("SPARK CORE / RDD VALIDATION PASSED")
print("=" * 45)

In [ ]:
delivered_rdd.unpersist()
orders_rdd.unpersist()
import time
time.sleep(int(os.getenv("FASTFOOD_UI_HOLD_SECONDS", "0")))
spark.stop()